In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns

In [ ]:
import warnings
warnings.filterwarnings("ignore")
sns.set(style="whitegrid")

In [ ]:
df=pd.read_csv("AIML Dataset.csv")

In [ ]:
df.head()


In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df["isFraud"].value_counts()

In [ ]:
df.isnull().sum()


In [ ]:
df.value_counts("type").plot(kind="bar",title="transactiontype",color="red")
plt.xlabel("transaction type")
plt.ylabel("count")
plt.show()

In [ ]:
fraudrate=df.groupby("type")["isFraud"].mean().sort_values(ascending=False)
fraudrate.plot(kind="bar",color="red",title="fraud rate")
plt.xlabel("transaction type")
plt.show()



In [ ]:
df["amount"].describe().astype(int)

In [ ]:
sns.histplot(np.log1p(df["amount"]),bins=100,color="red",kde=True)
plt.xlabel("Log of Amount")
plt.ylabel("Frequency")
plt.title("Distribution of Transaction Amounts")
plt.show()

In [ ]:
sns.boxplot(data=df[df["amount"]< 50000], x="isFraud", y="amount", palette="Set2" )
plt.show()

In [ ]:
df.columns


In [ ]:
df["balancedifferoriginal"]=df["oldbalanceOrg"]-df["newbalanceOrig"]
df["balancedifferdesti"]=df["newbalanceDest"]-df["oldbalanceDest"]


In [ ]:
(df["balancedifferoriginal"] < 0).sum()

In [ ]:
frauds_per_step=df[df["isFraud"] == 1]["step"].value_counts().sort_index()
sns.lineplot(x=frauds_per_step.index,y=frauds_per_step.values,color="red")
plt.show()

In [ ]:
top_senders=df["nameOrig"].value_counts().head(5)
top_receievers=df["nameDest"].value_counts().head(5)
sns.barplot(x=top_senders.index,y=top_senders.values,color="red")
plt.show()

In [ ]:
fraud_users=df[df["isFraud"]==1]["nameOrig"].value_counts().head(5)
 

In [ ]:
fraud_users


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [ ]:
df.head()


In [ ]:
df_model=df.drop(["step","nameOrig","nameDest","isFlaggedFraud"],axis=1)

In [ ]:
df_model.head()

In [ ]:
catogorical=['type']
numerical=["amount","oldbalanceOrg","newbalanceOrig","oldbalanceDest","newbalanceDest"] 


In [ ]:
y=df_model["isFraud"]
X=df_model.drop("isFraud",axis=1)

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,stratify=y)

In [ ]:
preprocessor=ColumnTransformer(
    transformers=[("nums",StandardScaler(),numerical),("cata",OneHotEncoder(drop="first"),catogorical)],remainder="drop"
)


In [ ]:
pipeline=Pipeline(steps=[("preprocessor",preprocessor),("model",LogisticRegression(class_weight="balanced",max_iter=1000))])

In [ ]:
pipeline.fit(X_train,y_train)

In [ ]:
y_predict=pipeline.predict(X_test)
print(classification_report(y_test,y_predict))

In [ ]:
confusion_matrix(y_test,y_predict)

In [ ]:
pipeline.score(X_test,y_test)


In [ ]:
import joblib
joblib.dump(pipeline,"fraud_detection_model.pkl")


In [ ]:
X.columns
